In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
import pickle
from google.colab import files

In [ ]:
uploaded = files.upload()  # Upload CSV
df = pd.read_csv("/content/synthetic_mobile_price_24k_2019_2025_platform.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Saving synthetic_mobile_price_24k_2019_2025_platform.csv to synthetic_mobile_price_24k_2019_2025_platform.csv
Dataset shape: (24000, 11)
Columns: ['platform', 'brand', 'phone_model', 'release_year', 'ram', 'storage', 'battery', 'camera', 'processor_score', 'condition', 'price']


,platform,brand,phone_model,release_year,ram,storage,battery,camera,processor_score,condition,price
0,Quikr,Realme,Realme 10,2023,16,256,5969,48,87,Good,122137
1,OLX,Vivo,Vivo V27,2020,12,128,5015,64,85,New,102395
2,Quikr,Samsung,Galaxy S24,2022,4,64,4274,48,92,Average,91091
3,Reliance Digital,OnePlus,OnePlus 11,2025,12,512,5328,48,71,New,290569
4,Amazon,Oppo,Reno 6,2022,4,512,5861,24,73,Like New,140283


In [ ]:
# Check first 5 rows
df.head()

# Check last 5 rows
df.tail()

# Check basic info
df.info()

# Check summary statistics
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   platform         24000 non-null  object
 1   brand            24000 non-null  object
 2   phone_model      24000 non-null  object
 3   release_year     24000 non-null  int64 
 4   ram              24000 non-null  int64 
 5   storage          24000 non-null  int64 
 6   battery          24000 non-null  int64 
 7   camera           24000 non-null  int64 
 8   processor_score  24000 non-null  int64 
 9   condition        24000 non-null  object
 10  price            24000 non-null  int64 
dtypes: int64(7), object(4)
memory usage: 2.0+ MB


,release_year,ram,storage,battery,camera,processor_score,price
count,24000.000000,24000.000000,24000.00000,24000.000000,24000.000000,24000.000000,24000.000000
mean,2021.988292,9.232417,239.62400,4897.973292,76.478333,81.941583,150983.319042
std,2.005864,4.317108,171.85464,635.448818,63.577844,10.071527,59884.736103
min,2019.000000,4.000000,64.00000,3800.000000,12.000000,65.000000,35414.000000
25%,2020.000000,6.000000,64.00000,4352.000000,24.000000,73.000000,107963.750000
50%,2022.000000,8.000000,128.00000,4893.000000,64.000000,82.000000,139643.500000
75%,2024.000000,12.000000,512.00000,5446.000000,108.000000,91.000000,181281.500000
max,2025.000000,16.000000,512.00000,5999.000000,200.000000,99.000000,538046.000000


In [ ]:
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
df = df.drop_duplicates()


Missing values:
 platform           0
brand              0
phone_model        0
release_year       0
ram                0
storage            0
battery            0
camera             0
processor_score    0
condition          0
price              0
dtype: int64

Duplicates: 0


In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = df.select_dtypes(exclude="object").columns.tolist()
print("\nCATEGORICAL COLUMNS:", cat_cols)
print("NUMERICAL COLUMNS:", num_cols)


CATEGORICAL COLUMNS: ['platform', 'brand', 'phone_model', 'condition']
NUMERICAL COLUMNS: ['release_year', 'ram', 'storage', 'battery', 'camera', 'processor_score', 'price']


In [ ]:
brand_encoder = LabelEncoder()
condition_encoder = LabelEncoder()
platform_encoder = LabelEncoder()
phone_model_encoder = LabelEncoder()

df["brand"] = brand_encoder.fit_transform(df["brand"])
df["condition"] = condition_encoder.fit_transform(df["condition"])
df["platform"] = platform_encoder.fit_transform(df["platform"])
df["phone_model"] = phone_model_encoder.fit_transform(df["phone_model"])

In [ ]:
TARGET = "price"
X = df.drop(TARGET, axis=1)
y = df[TARGET]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

Train size: (19200, 10)
Test size : (4800, 10)


In [ ]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, train_pred)
train_mape = mean_absolute_percentage_error(y_train, train_pred)

print(" TRAINING PHASE ACCURACY")
print(f"R² Score   : {train_r2:.4f}")
print(f"MAPE       : {train_mape:.4f}")  # lower is better
print("-"*40)

 TRAINING PHASE ACCURACY
R² Score   : 0.9981
MAPE       : 0.0144
----------------------------------------


In [ ]:
#Save Model
# -------------------------------
model.save_model("xgboost_mobile_price_model.json")
print(" Model saved as xgboost_mobile_price_model.json")

# Optional: Save encoders
with open("encoders.pkl", "wb") as f:
    pickle.dump({
        "brand_encoder": brand_encoder,
        "condition_encoder": condition_encoder,
        "platform_encoder": platform_encoder,
        "phone_model_encoder": phone_model_encoder
    }, f)
print(" Encoders saved as encoders.pkl")

 Model saved as xgboost_mobile_price_model.json
 Encoders saved as encoders.pkl


In [ ]:
 #Load model
loaded_model = XGBRegressor()
loaded_model.load_model("xgboost_mobile_price_model.json")
print(" Model reloaded successfully")

# Load encoders
with open("encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

brand_encoder = encoders["brand_encoder"]
condition_encoder = encoders["condition_encoder"]
platform_encoder = encoders["platform_encoder"]
phone_model_encoder = encoders["phone_model_encoder"]

 Model reloaded successfully


In [ ]:
def predict_price_with_range_fallback(brand_str, model_str, condition_str, ram, storage):
    # Encode categorical safely
    try:
        brand_enc = brand_encoder.transform([brand_str])[0]
    except ValueError:
        brand_enc = int(df["brand"].median())

    try:
        model_enc = phone_model_encoder.transform([model_str])[0]
    except ValueError:
        model_enc = int(df["phone_model"].median())

    try:
        condition_enc = condition_encoder.transform([condition_str])[0]
    except ValueError:
        condition_enc = int(df["condition"].median())

    platform_enc = 0  # default platform index

    # Auto-fill remaining features with median values
    release_year = int(df["release_year"].median())
    battery = int(df["battery"].median())
    camera = int(df["camera"].median())
    processor_score = int(df["processor_score"].median())

    # -----------------------------
    # Prepare input dataframe
    # -----------------------------
    input_df = pd.DataFrame({
        "platform": [platform_enc],
        "brand": [brand_enc],
        "phone_model": [model_enc],
        "release_year": [release_year],
        "ram": [ram],
        "storage": [storage],
        "battery": [battery],
        "camera": [camera],
        "processor_score": [processor_score],
        "condition": [condition_enc]
    })

    # -----------------------------
    # CONVERT ALL COLUMNS TO NUMERIC
    # -----------------------------
    input_df = input_df.astype(float)   # <---- HERE

    # -----------------------------
    # Predict price
    # -----------------------------
    base_price = loaded_model.predict(input_df)[0]
    lower = int(base_price * 0.95)
    upper = int(base_price * 1.05)

    return int(base_price), lower, upper

In [ ]:
def predict_price_with_range_fallback(brand_str, model_str, condition_str, ram, storage):
    # Encode categorical safely
    try:
        brand_enc = brand_encoder.transform([brand_str])[0]
    except ValueError:
        brand_enc = int(df["brand"].median())

    try:
        model_enc = phone_model_encoder.transform([model_str])[0]
    except ValueError:
        model_enc = int(df["phone_model"].median())

    try:
        condition_enc = condition_encoder.transform([condition_str])[0]
    except ValueError:
        condition_enc = int(df["condition"].median())

    platform_enc = 0  # default platform index

    # Auto-fill remaining features with median values
    release_year = int(df["release_year"].median())
    battery = int(df["battery"].median())
    camera = int(df["camera"].median())
    processor_score = int(df["processor_score"].median())

    # -----------------------------
    # Prepare input dataframe
    # -----------------------------
    input_df = pd.DataFrame({
        "platform": [platform_enc],
        "brand": [brand_enc],
        "phone_model": [model_enc],
        "release_year": [release_year],
        "ram": [ram],
        "storage": [storage],
        "battery": [battery],
        "camera": [camera],
        "processor_score": [processor_score],
        "condition": [condition_enc]
    })

    # -----------------------------
    # CONVERT ALL COLUMNS TO NUMERIC
    # -----------------------------
    input_df = input_df.astype(float)   # <---- HERE

    # -----------------------------
    # Predict price
    # -----------------------------
    base_price = loaded_model.predict(input_df)[0]
    lower = int(base_price * 0.95)
    upper = int(base_price * 1.05)

    return int(base_price), lower, upper

print("\n----- ENTER MOBILE DETAILS -----")
brand = input("Brand: ").title()
model_name = input("Phone Model: ").title()
condition = input("Condition (New / Used / Refurbished): ").title()
ram = int(input("RAM (GB): "))
storage = int(input("Storage (GB): "))

# Call the fallback prediction function
price, low, high = predict_price_with_range_fallback(brand, model_name, condition, ram, storage)

print("\n📱 MOBILE PRICE PREDICTION")
print(f"Estimated Price     : ₹{price}")
print(f"Expected Price Range: ₹{low} – ₹{high}")


----- ENTER MOBILE DETAILS -----
Brand: Oppo 
Phone Model: Reno 6
Condition (New / Used / Refurbished): Used
RAM (GB): 4
Storage (GB): 64

📱 MOBILE PRICE PREDICTION
Estimated Price     : ₹106925
Expected Price Range: ₹101579 – ₹112271
